# IIR-Robust-Retrieval — Dataset Downloader & Verification (B0 Baseline)

**Author / Responsible:** Hoang (Data & Evaluation Preparation)  
**Team:** FUAVX (Phuong · Nguyen · Hoang)  
**Branch:** `b0-data-preparation`  

---

### Purpose
This Kaggle notebook automates:
1. Downloading official benchmark datasets (**Oxford5k**, **Paris6k**) and extracting image archives.
2. Downloading the custom pretrained **Caffe VGG16 weights** (`imagenet-caffe-vgg16-features-d369c8e.pth`).
3. Downloading **`retrieval-SfM-120k`** metadata and compressed image archives (`ims.tar.gz`).
4. Running the automated audit script `scripts/verify_datasets.py` to produce certified verification evidence.

### Kaggle Notebook Setup Requirements
- **Internet:** Must be toggled **ON** (in right-hand sidebar Settings).
- **Accelerator:** CPU is sufficient (no GPU required for download/verification).

In [ ]:
# Step 0: Clone the team repository
!git clone -b b0-data-preparation https://github.com/FUAVX/IIR-Robust-Retrieval.git /kaggle/working/IIR-Robust-Retrieval
%cd /kaggle/working/IIR-Robust-Retrieval

## 1. Download & Extract Evaluation Benchmarks (Oxford5k & Paris6k)

- **Oxford5k:** ~5,063 images, 55 query landmarks, ground truth `gnd_oxford5k.pkl`.
- **Paris6k:** ~6,412 images, 55 query landmarks, ground truth `gnd_paris6k.pkl`.

In [ ]:
import os

DATA_ROOT = "/kaggle/working/data"

print("=== DOWNLOADING OXFORD5K & PARIS6K ===")
!mkdir -p {DATA_ROOT}/test/oxford5k/jpg
!mkdir -p {DATA_ROOT}/test/paris6k/jpg

# 1. Oxford5k
print(">> Downloading Oxford5k images (~1.9 GB)...")
!wget -q --show-progress -c https://www.robots.ox.ac.uk/~vgg/data/oxbuildings/oxbuild_images-v1.tgz -P /kaggle/working/
print(">> Extracting Oxford5k images...")
!tar -xzf /kaggle/working/oxbuild_images-v1.tgz -C {DATA_ROOT}/test/oxford5k/jpg/
!rm -f /kaggle/working/oxbuild_images-v1.tgz

print(">> Downloading Oxford5k ground truth pkl...")
!wget -q -c http://cmp.felk.cvut.cz/cnnimageretrieval/data/test/oxford5k/gnd_oxford5k.pkl -P {DATA_ROOT}/test/oxford5k/

# 2. Paris6k
print(">> Downloading Paris6k images (~2.0 GB)...")
!wget -q --show-progress -c https://www.robots.ox.ac.uk/~vgg/data/parisbuildings/paris_1-v1.tgz -P /kaggle/working/
!wget -q --show-progress -c https://www.robots.ox.ac.uk/~vgg/data/parisbuildings/paris_2-v1.tgz -P /kaggle/working/
print(">> Extracting Paris6k images...")
!tar -xzf /kaggle/working/paris_1-v1.tgz -C {DATA_ROOT}/test/paris6k/jpg/
!tar -xzf /kaggle/working/paris_2-v1.tgz -C {DATA_ROOT}/test/paris6k/jpg/
!rm -f /kaggle/working/paris_1-v1.tgz /kaggle/working/paris_2-v1.tgz

print(">> Downloading Paris6k ground truth pkl...")
!wget -q -c http://cmp.felk.cvut.cz/cnnimageretrieval/data/test/paris6k/gnd_paris6k.pkl -P {DATA_ROOT}/test/paris6k/

print(">> Oxford5k & Paris6k download complete!")

## 2. Download Pretrained Caffe VGG16 Backbone Weights

- File: `imagenet-caffe-vgg16-features-d369c8e.pth` (~58 MB)
- **CRITICAL:** Do NOT substitute torchvision default ImageNet weights.

In [ ]:
print("=== DOWNLOADING PRETRAINED CAFFE VGG16 WEIGHTS ===")
!mkdir -p {DATA_ROOT}/networks
!wget -q --show-progress -c http://cmp.felk.cvut.cz/cnnimageretrieval/data/networks/imagenet/imagenet-caffe-vgg16-features-d369c8e.pth -P {DATA_ROOT}/networks/
print(">> Weights downloaded successfully!")

## 3. Download `retrieval-SfM-120k` Training Supervision & Image Archive

- Metadata: `retrieval-SfM-120k.pkl` (~12 MB), `retrieval-SfM-120k-whiten.pkl` (~3 MB)
- Image archive: `ims.tar.gz` (~11.5 GB compressed)

In [ ]:
print("=== DOWNLOADING RETRIEVAL-SFM-120K ===")
!mkdir -p {DATA_ROOT}/train/retrieval-SfM-120k

print(">> Downloading metadata pickles...")
!wget -q -c http://cmp.felk.cvut.cz/cnnimageretrieval/data/train/dbs/retrieval-SfM-120k.pkl -P {DATA_ROOT}/train/retrieval-SfM-120k/
!wget -q -c http://cmp.felk.cvut.cz/cnnimageretrieval/data/train/dbs/retrieval-SfM-120k-whiten.pkl -P {DATA_ROOT}/train/retrieval-SfM-120k/

print(">> Downloading image archive ims.tar.gz (~11.5 GB)... (estimated 2-4 minutes on Kaggle network)")
!wget -q --show-progress -c http://cmp.felk.cvut.cz/cnnimageretrieval/data/train/ims/ims.tar.gz -P {DATA_ROOT}/train/retrieval-SfM-120k/

print(">> SfM-120k download complete!")

## 4. Run Automated Dataset Verification Audit

Executes `scripts/verify_datasets.py` to audit metadata integrity, verify positive/junk structures, check bounding boxes, and decode test sample images.

In [ ]:
# Execute verification script against the downloaded DATA_ROOT
!python3 scripts/verify_datasets.py --data-root {DATA_ROOT} --sample-size 20

## 5. Optional: Generate Cluster-Preserved Subset for Rapid Testing (C1/C3)

To enable Phuong to quickly test the training loop without loading 120k images, create a cluster-safe 50-cluster subset:

In [ ]:
!python3 scripts/create_sfm_subset.py \
    --input-pkl {DATA_ROOT}/train/retrieval-SfM-120k/retrieval-SfM-120k.pkl \
    --output-pkl {DATA_ROOT}/train/retrieval-SfM-120k/retrieval-SfM-subset50.pkl \
    --num-clusters 50 \
    --seed 42

## 6. How to Save as a Kaggle Dataset for the Team

1. Click **"Save Version"** (top right) -> Select **"Quick Save"**.
2. Once saved, go to the notebook's **Output** tab.
3. Click **"New Dataset"**.
4. Enter dataset title: `iir-robust-retrieval-datasets`.
5. Share the dataset link or path with Phuong (`/kaggle/input/iir-robust-retrieval-datasets/`).